In [3]:
import re

class Term:
    def __init__(self, name, args=None):
        self.name = name
        self.args = args if args is not None else []

    def is_variable(self):
        return len(self.args) == 0 and self.name[0].islower()

    def is_constant(self):
        return len(self.args) == 0 and self.name[0].isupper()

    def is_compound(self):
        return len(self.args) > 0

    def __repr__(self):
        if self.is_compound():
            return f"{self.name}({', '.join(map(str, self.args))})"
        return self.name

    def __eq__(self, other):
        if not isinstance(other, Term):
            return False
        return self.name == other.name and self.args == other.args

def parse_term(s):
    s = s.replace(" ", "")
    if not s:
        raise ValueError("Empty expression")

    match = re.match(r'^([a-zA-Z0-9]+)$', s)
    if match:
        return Term(match.group(1))

    match = re.match(r'^([a-zA-Z0-9]+)\((.*)\)$', s)
    if match:
        name = match.group(1)
        args_str = match.group(2)
        args = []

        bracket_level = 0
        current_arg = []
        for char in args_str:
            if char == '(':
                bracket_level += 1
                current_arg.append(char)
            elif char == ')':
                bracket_level -= 1
                current_arg.append(char)
            elif char == ',' and bracket_level == 0:
                args.append(parse_term("".join(current_arg)))
                current_arg = []
            else:
                current_arg.append(char)
        if current_arg:
            args.append(parse_term("".join(current_arg)))

        return Term(name, args)

    raise ValueError(f"Failed to parse term: {s}")

def occurs_check(var, term, subst):
    if var == term:
        return True
    elif term.is_variable():
        if term.name in subst:
            return occurs_check(var, subst[term.name], subst)
        return False
    elif term.is_compound():
        return any(occurs_check(var, arg, subst) for arg in term.args)
    return False

def apply_subst(term, subst):
    if term.is_variable():
        if term.name in subst:
            return apply_subst(subst[term.name], subst)
        return term
    elif term.is_compound():
        return Term(term.name, [apply_subst(arg, subst) for arg in term.args])
    return term

def unify(x, y, subst=None):
    if subst is None:
        subst = {}

    x = apply_subst(x, subst)
    y = apply_subst(y, subst)

    if x == y:
        return subst

    if x.is_variable():
        return unify_var(x, y, subst)

    if y.is_variable():
        return unify_var(y, x, subst)

    if x.is_compound() and y.is_compound():
        if x.name != y.name or len(x.args) != len(y.args):
            return None
        for arg_x, arg_y in zip(x.args, y.args):
            subst = unify(arg_x, arg_y, subst)
            if subst is None:
                return None
        return subst

    return None

def unify_var(var, term, subst):
    if var.name in subst:
        return unify(subst[var.name], term, subst)
    if term.is_variable() and term.name in subst:
        return unify(var, subst[term.name], subst)
    if occurs_check(var, term, subst):
        print(f"Occur check failed: {var} is inside {term}")
        return None

    new_subst = subst.copy()
    new_subst[var.name] = term
    for k, v in list(new_subst.items()):
        if k != var.name:
            new_subst[k] = apply_subst(v, {var.name: term})
    return new_subst


print("\n=== UNIFIED LOGIC EXPRESSIONS ===")
try:
    user_t1 = input("Enter first term (default: f(x, g(y))): ").strip() or "f(x, g(y))"
    user_t2 = input("Enter second term (default: f(A, g(z))): ").strip() or "f(A, g(z))"
    t1 = parse_term(user_t1)
    t2 = parse_term(user_t2)
    result = unify(t1, t2)
    print(f"\nTerm 1 parsed: {t1}")
    print(f"Term 2 parsed: {t2}")
    print(f"Resulting MGU: {result}")
    if result is not None:
        print(f"Unified Expression: {apply_subst(t1, result)}")
except Exception as e:
    print(f"Error processing terms: {e}")


=== UNIFIED LOGIC EXPRESSIONS ===
Enter first term (default: f(x, g(y))): EAT(x, MANGO)
Enter second term (default: f(A, g(z))): EAT(ADITYA, y)

Term 1 parsed: EAT(x, MANGO)
Term 2 parsed: EAT(ADITYA, y)
Resulting MGU: {'x': ADITYA, 'y': MANGO}
Unified Expression: EAT(ADITYA, MANGO)
